# Severity model: sentence-transformer + CatBoost

Trains a model that reads **only the complaint description** and predicts **Low / Medium / High / Critical**, using the LLM-labelled file `llm_labels.csv` (58,640 complaints). Everything runs live in this notebook, with training progress printed under the cell.

**Before you run (Colab):** Runtime -> Change runtime type -> **T4 GPU**. A GPU makes the embedding step about a minute and CatBoost training a few minutes. Without a GPU it still works but is much slower, so the notebook trains fewer iterations.

**What you get at the end** (in the `severity_model/` folder, also zipped and downloaded):
- `severity_catboost.cbm`: the trained CatBoost model (portable native format)
- `severity_catboost.pkl`: the same model as a pickle
- `label_map.json`: class names, encoder name and settings
- `severity_predictor.py`: a small self-contained class, so any Python system can call the model
- `metrics.json`: test-set results

## 1. Setup

In [ ]:
import importlib.util, subprocess, sys
for pkg, mod in [("sentence-transformers", "sentence_transformers"), ("catboost", "catboost")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("packages ready")

In [ ]:
SMOKE_TEST = False   # True = tiny quick run (3,000 rows, 60 iterations) only to check the notebook works
SEED = 42
REQUIRE_GPU = True    # stops early if no GPU is attached (CPU training here takes hours); set False to force a slow CPU run
ENCODER = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"   # multilingual: handles English and Tamil
MAX_SEQ_LEN = 128                                                          # covers ~99% of complaints (median ~120 characters)
# Raw URL of labelling/output/llm_labels.csv. Change the branch name in it if the branch has been merged or renamed, or
# leave it failing and upload the file when asked (or place llm_labels.csv next to the notebook).
DATA_URL = "https://raw.githubusercontent.com/Shreelakshmigbhat/Citizen-Complaint-Resolution-System/feat/severity-labelling-rubric/severity-service/labelling/output/llm_labels.csv"
OUT_DIR = "severity_model"

## 2. Load the labelled data

In [ ]:
import io
from pathlib import Path
import numpy as np, pandas as pd

local = [p for p in (Path("llm_labels.csv"), Path("../labelling/output/llm_labels.csv"),
                     Path("severity-service/labelling/output/llm_labels.csv")) if p.exists()]
if local:
    df = pd.read_csv(local[0]); print("loaded", local[0])
else:
    try:
        df = pd.read_csv(DATA_URL); print("downloaded from GitHub")
    except Exception as e:                      # private repo or offline: upload the file instead
        print("download failed (", type(e).__name__, "): upload llm_labels.csv")
        from google.colab import files
        up = files.upload(); df = pd.read_csv(io.BytesIO(next(iter(up.values()))))
if SMOKE_TEST:
    df = df.sample(3000, random_state=SEED).reset_index(drop=True)
print(df.shape); df.head(3)

## 3. Clean the text and look at the classes
The same cleaning is used at prediction time (it is built into `severity_predictor.py`), so training and serving see identical text.

In [ ]:
import html, re
import matplotlib.pyplot as plt

def text_for_model(raw):
    if not isinstance(raw, str): return ""
    s = html.unescape(html.unescape(raw))
    s = re.sub(r"<[^>]+>", " ", s)
    s = s.replace("\u200b", " ").replace("\ufeff", " ")
    s = re.sub(r"[\r\n\t]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip(" .,;:-}{").strip()
    s = s.split("@", 1)[0] if "@" in s else s
    s = re.sub(r"\b\d{4}-[0-9A-Z]{6}\b", " ", s)
    s = re.sub(r"\+?\d[\d\s-]{8,}\d", " ", s)
    return re.sub(r"\s+", " ", s).strip()

LABELS = ["Low", "Medium", "High", "Critical"]
df["text"] = df["description"].map(text_for_model)
df = df[df["text"].str.len() > 0].reset_index(drop=True)
y = df["label"].map({l: i for i, l in enumerate(LABELS)}).astype(int).to_numpy()

counts = df["label"].value_counts().reindex(LABELS)
print(counts.to_string()); print("share:", (counts / len(df) * 100).round(1).to_dict())
counts.plot(kind="bar", color=["#8fbf8f", "#e7c46a", "#e8955a", "#d9534f"], figsize=(5, 3), title="Label counts"); plt.xticks(rotation=0); plt.show()

## 4. Turn each description into a 384-number embedding
The multilingual MiniLM sentence-transformer maps text to a vector where similar meanings are close. Those 384 numbers are the only features CatBoost sees.

In [ ]:
import time, torch, hashlib
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
if device == "cpu" and REQUIRE_GPU and not SMOKE_TEST:
    raise RuntimeError("No GPU attached. In Colab: Runtime -> Change runtime type -> T4 GPU (in VS Code: Select Kernel -> Colab -> New Colab Server -> choose a GPU), "
                       "then run again. To force a (very slow) CPU run, set REQUIRE_GPU = False in the settings cell.")

texts = df["text"].tolist()
key = hashlib.sha1((ENCODER + str(MAX_SEQ_LEN) + "|".join(texts)).encode("utf-8")).hexdigest()[:16]
cache = Path(f"emb_{key}.npy")
if cache.exists():
    emb = np.load(cache); print("loaded saved embeddings", cache.name, emb.shape)
else:
    encoder = SentenceTransformer(ENCODER, device=device)
    encoder.max_seq_length = MAX_SEQ_LEN
    t0 = time.time()
    emb = encoder.encode(texts, normalize_embeddings=True, batch_size=128 if device == "cuda" else 64, show_progress_bar=True).astype(np.float32)
    print(f"encoded {len(emb)} complaints in {time.time() - t0:.0f}s ->", emb.shape)
    np.save(cache, emb)                      # a rerun or restart skips the encoder
FEATURES = [f"e{i}" for i in range(emb.shape[1])]
X = pd.DataFrame(emb, columns=FEATURES)

## 5. Train and check with 5-fold stratified cross-validation
The complaints are split into **5 folds** that each keep the same class mix. Five models are trained; each one is scored on the fold it never saw. So **every complaint gets one honest prediction** from a model that did not train on it, and you get a mean and a spread across folds instead of one lucky or unlucky split.

Inside each fold, 10% of the training part is set aside to decide when to stop training (early stopping), so the scored fold is never used for any choice.

Settings chosen for this data:
- **MultiClass loss**, 4 classes.
- **`auto_class_weights="SqrtBalanced"`**: Critical is only ~10% of the data, and missing a Critical complaint is the costliest mistake, so rarer classes get a gentler boost in weight.
- **depth 6, learning rate 0.1, L2 regularisation 5**: moderate capacity for 384 dense, normalised features, with enough regularisation not to memorise noisy LLM labels.
- **Up to 2,000 trees with early stopping (100 rounds)**; the best iteration is kept.

In [ ]:
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, cohen_kappa_score,
                             classification_report, confusion_matrix, log_loss)

N_FOLDS = 5
USE_GPU = torch.cuda.is_available()
params = dict(
    loss_function="MultiClass", eval_metric="MultiClass",
    iterations=(60 if SMOKE_TEST else 2000) if USE_GPU or SMOKE_TEST else 300,
    learning_rate=0.1, depth=6, l2_leaf_reg=5,
    auto_class_weights="SqrtBalanced", early_stopping_rounds=100,
    random_seed=SEED, verbose=0,
)
if USE_GPU:
    params.update(task_type="GPU", devices="0")
else:
    params.update(task_type="CPU", thread_count=-1)
    print("No GPU: limiting to", params["iterations"], "trees (CPU training on 47k x 384 features is slow on Colab).")

def score(yt, pred, proba=None):
    rep = classification_report(yt, pred, labels=range(4), target_names=LABELS, output_dict=True, zero_division=0)
    s = {"accuracy": accuracy_score(yt, pred), "balanced_accuracy": balanced_accuracy_score(yt, pred),
         "macro_f1": f1_score(yt, pred, average="macro"), "weighted_f1": f1_score(yt, pred, average="weighted"),
         "quadratic_kappa": cohen_kappa_score(yt, pred, weights="quadratic"),
         "within_one_level": float((np.abs(yt - pred) <= 1).mean()), "critical_recall": rep["Critical"]["recall"]}
    if proba is not None:
        s["log_loss"] = log_loss(yt, proba, labels=range(4))
    return s, rep

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof = np.zeros((len(df), 4))              # out-of-fold probabilities: one honest prediction per complaint
fold_rows, best_iters, curve = [], [], None
t_all = time.time()

for k, (tr_all, te) in enumerate(skf.split(X, y), 1):
    tr, va = train_test_split(tr_all, test_size=0.1, stratify=y[tr_all], random_state=SEED)
    t0 = time.time()
    m = CatBoostClassifier(**params)
    m.fit(Pool(X.iloc[tr], y[tr]), eval_set=Pool(X.iloc[va], y[va]), use_best_model=True)
    p = m.predict_proba(X.iloc[te]); oof[te] = p
    s, _ = score(y[te], p.argmax(1), p)
    s.update(fold=k, best_iteration=int(m.get_best_iteration()), n_train=len(tr), n_val=len(va), n_test=len(te))
    fold_rows.append(s); best_iters.append(m.get_best_iteration())
    if curve is None:
        curve = (m.get_evals_result(), m.get_best_iteration())
    print(f"fold {k}/{N_FOLDS}: accuracy {s['accuracy']:.3f} | macro F1 {s['macro_f1']:.3f} | Critical recall {s['critical_recall']:.3f} "
          f"| best iteration {s['best_iteration']} | {time.time() - t0:.0f}s", flush=True)
print(f"\nall folds done in {(time.time() - t_all) / 60:.1f} min")

## 6. Cross-validation results

In [ ]:
folds = pd.DataFrame(fold_rows).set_index("fold")
cols = ["accuracy", "balanced_accuracy", "macro_f1", "quadratic_kappa", "within_one_level", "critical_recall", "log_loss", "best_iteration"]
summary = pd.DataFrame({"mean": folds[cols].mean(), "std": folds[cols].std()})
display(folds[cols].round(4)); display(summary.round(4))

yp = oof.argmax(1)
oof_scores, rep = score(y, yp, oof)
print(f"\nOut-of-fold over all {len(y)} complaints: accuracy {oof_scores['accuracy']:.3f} | macro F1 {oof_scores['macro_f1']:.3f} | "
      f"quadratic kappa {oof_scores['quadratic_kappa']:.3f} | within one level {oof_scores['within_one_level']:.3f} | "
      f"Critical recall {oof_scores['critical_recall']:.3f}\n")
print(classification_report(y, yp, labels=range(4), target_names=LABELS, digits=3))

In [ ]:
cm = confusion_matrix(y, yp, labels=range(4))
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for a, data, title, fmt in [(ax[0], cm, "Counts", "{:d}"), (ax[1], cm / cm.sum(1, keepdims=True), "Row % (share of each true class)", "{:.0%}")]:
    a.imshow(data, cmap="Blues")
    a.set_xticks(range(4)); a.set_yticks(range(4)); a.set_xticklabels(LABELS); a.set_yticklabels(LABELS)
    a.set_xlabel("predicted"); a.set_ylabel("LLM label (true)"); a.set_title(f"Out-of-fold confusion matrix: {title}")
    for i in range(4):
        for j in range(4):
            a.text(j, i, fmt.format(data[i, j]), ha="center", va="center", color="white" if data[i, j] > data.max() / 2 else "black")
plt.tight_layout(); plt.show()

ev, bi = curve
plt.figure(figsize=(6, 3.5))
plt.plot(ev["learn"]["MultiClass"], label="train"); plt.plot(ev["validation"]["MultiClass"], label="validation")
plt.axvline(bi, color="grey", ls="--", label="best iteration")
plt.xlabel("iteration"); plt.ylabel("log loss"); plt.legend(); plt.title("Training curve (fold 1)"); plt.tight_layout(); plt.show()

> **How to read these numbers:** the labels this model learns from were written by an LLM, so the scores say how well the model reproduces **those** labels, not how right they are about real severity. Earlier checks found the LLM labels agree with the original labels only about 60% of the time, so treat this as a first model.

## 7. Train the final model on all the data
The five fold models were only for measuring. The model you plug in is trained once on **all** the complaints, with the number of trees set from the cross-validation (the mean best iteration, plus 10% because it sees more data).

In [ ]:
final_iters = int(round(np.mean(best_iters) * 1.1))
final_params = {k: v for k, v in params.items() if k != "early_stopping_rounds"}
final_params.update(iterations=final_iters, verbose=100)
model = CatBoostClassifier(**final_params)
t0 = time.time()
model.fit(Pool(X, y))
print(f"final model: {final_iters} trees on {len(y)} complaints in {time.time() - t0:.0f}s")

metrics = {
    "cross_validation": {"n_folds": N_FOLDS, "per_fold": fold_rows,
                         "mean": summary["mean"].to_dict(), "std": summary["std"].to_dict()},
    "out_of_fold_all_rows": {**oof_scores, "per_class": {l: {k: rep[l][k] for k in ("precision", "recall", "f1-score", "support")} for l in LABELS}},
    "final_model": {"trained_on_rows": int(len(y)), "iterations": final_iters},
}

## 8. Save the model so any system can use it
Writes `severity_catboost.cbm` (portable), a pickle copy, the label map, the metrics and a small predictor class, then zips them.

In [ ]:
import json, pickle
out = Path(OUT_DIR); out.mkdir(exist_ok=True)
model.save_model(str(out / "severity_catboost.cbm"))
with open(out / "severity_catboost.pkl", "wb") as f:
    pickle.dump(model, f)
(out / "label_map.json").write_text(json.dumps({
    "labels": LABELS, "embedding_model": ENCODER, "max_seq_length": MAX_SEQ_LEN,
    "features": "e0..e383 normalised sentence embeddings of the cleaned description (no other inputs)",
    "trained_on": "llm_labels.csv (LLM severity labels, rubric v0.3); final model trained on all rows after 5-fold CV",
}, indent=1), "utf-8")
(out / "metrics.json").write_text(json.dumps(metrics, indent=1, default=float), "utf-8")
print(sorted(p.name for p in out.iterdir()))

In [ ]:
%%writefile severity_model/severity_predictor.py
"""Severity predictor: complaint description -> Low / Medium / High / Critical.

Sentence-transformer embeddings + a CatBoost classifier. Self-contained: it needs only
this file, `severity_catboost.cbm` and `label_map.json` in the same folder, plus
`pip install catboost sentence-transformers`.

    from severity_predictor import SeverityModel
    model = SeverityModel(".")                       # folder holding the .cbm and label_map.json
    model.predict(["Open manhole near the school, children could fall in"])
    # [{'severity': 'Critical', 'severity_id': 3, 'confidence': 0.81, 'probabilities': {...}}]
"""
import html
import json
import re
from pathlib import Path

import numpy as np
from catboost import CatBoostClassifier


def text_for_model(raw: str) -> str:
    """Same cleaning that was applied to the training text."""
    if not isinstance(raw, str):
        return ""
    s = html.unescape(html.unescape(raw))
    s = re.sub(r"<[^>]+>", " ", s)
    s = s.replace("​", " ").replace("﻿", " ")
    s = re.sub(r"[\r\n\t]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip(" .,;:-}{").strip()
    s = s.split("@", 1)[0] if "@" in s else s
    s = re.sub(r"\b\d{4}-[0-9A-Z]{6}\b", " ", s)
    s = re.sub(r"\+?\d[\d\s-]{8,}\d", " ", s)
    return re.sub(r"\s+", " ", s).strip()


class SeverityModel:
    def __init__(self, model_dir: str = ".", device: str | None = None):
        d = Path(model_dir)
        meta = json.loads((d / "label_map.json").read_text("utf-8"))
        self.labels: list[str] = meta["labels"]
        self.clf = CatBoostClassifier()
        self.clf.load_model(str(d / "severity_catboost.cbm"))
        if len(self.labels) != len(self.clf.classes_):
            raise RuntimeError("label_map.json does not match the model's classes")
        from sentence_transformers import SentenceTransformer

        self.encoder = SentenceTransformer(meta["embedding_model"], device=device)
        self.encoder.max_seq_length = meta.get("max_seq_length", 128)

    def predict_proba(self, texts: list[str]) -> np.ndarray:
        emb = self.encoder.encode([text_for_model(t) for t in texts], normalize_embeddings=True, batch_size=64)
        return self.clf.predict_proba(np.asarray(emb, dtype=np.float32))

    def predict(self, texts: list[str]) -> list[dict]:
        """One dict per input text: severity, severity_id, confidence and all class probabilities."""
        out = []
        for p in self.predict_proba(texts):
            k = int(np.argmax(p))
            out.append({
                "severity": self.labels[k],
                "severity_id": k,
                "confidence": round(float(p[k]), 4),
                "probabilities": {l: round(float(x), 4) for l, x in zip(self.labels, p)},
            })
        return out


if __name__ == "__main__":
    import sys

    m = SeverityModel(Path(__file__).resolve().parent)
    for t, r in zip(sys.argv[1:], m.predict(sys.argv[1:])):
        print(f"{r['severity']:9s} {r['confidence']:.2f}  {t}")


In [ ]:
import shutil
zip_path = shutil.make_archive("severity_model", "zip", OUT_DIR)
print("bundle:", zip_path, f"({Path(zip_path).stat().st_size / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(zip_path)            # browser Colab: saves the zip to your computer
except Exception:
    print("Automatic download is not available here. The files are on the Colab server at:",
          str(Path(zip_path).resolve()), "and", str(Path(OUT_DIR).resolve()))

## 9. Try it exactly as another system would
A fresh load from the saved files, calling the predictor class.

In [ ]:
import sys
sys.path.insert(0, OUT_DIR)
from severity_predictor import SeverityModel

severity = SeverityModel(OUT_DIR)
examples = [
    "Open manhole on the main road near the school, children could fall in",
    "Street dog bit a child this morning, please take action",
    "Stagnant water in our street breeding mosquitoes for two weeks",
    "Street light not working in 3rd cross street",
    "Please remove the illegal advertisement banner near the bus stop",
    "How is property tax calculated for a commercial building?",
]
for text, r in zip(examples, severity.predict(examples)):
    print(f"{r['severity']:9s} {r['confidence']:.2f}  {text}")

## Using the model elsewhere
```python
# pip install catboost sentence-transformers
from severity_predictor import SeverityModel
model = SeverityModel("severity_model")          # folder with severity_catboost.cbm + label_map.json
model.predict(["Open manhole near the school"])   # -> [{'severity': 'Critical', 'confidence': 0.81, 'probabilities': {...}, ...}]
```
The `.cbm` file can also be loaded directly with `CatBoostClassifier().load_model(...)`; its input is the 384 normalised embedding values `e0..e383` produced by the encoder named in `label_map.json`.